# Практична робота 3. Геометричні характеристики та матеріаломісткість кронштейна

## Мета
Навчитися за параметричною моделлю визначати:
- об'єм основного корпусу;
- об'єм ребер жорсткості;
- об'єм матеріалу, що припадає на отвори;
- сумарний об'єм деталі;
- масу деталі для різних матеріалів;
- вплив конструктивних змін на масу.

**Важливо про точність моделі.** У попередній роботі отвори відображалися лише контурами. У цій роботі вони враховуються в розрахунку як наскрізні циліндричні отвори. Розрахунок є аналітичним наближенням для ідеалізованої геометрії. Він не враховує фаски, заокруглення, зварні шви, технологічні припуски та реальні допуски.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

plt.rcParams['figure.figsize'] = (10, 5)


## 1. Геометрична схема розрахунку

### Основний корпус

Поперечний L-подібний профіль складається з двох прямокутників із відніманням спільної ділянки:

\[
A_{L}=DT+HT-T^2=T(D+H-T).
\]

Об'єм основного корпусу:

\[
V_{корп}=A_L W=T(D+H-T)W.
\]

### Ребра жорсткості

Кожне ребро — трикутна призма з катетом `G` і товщиною `Rt`:

\[
V_{ребра}=\frac{G^2}{2}Rt.
\]

Для двох ребер:

\[
V_{ребер}=G^2Rt.
\]

### Отвори

У моделі є `n` отворів на горизонтальній полиці та `n` — на вертикальній. Усього `2n` наскрізних отворів діаметра `d`, кожен проходить крізь товщину `T`:

\[
V_{отворів}=2n\pi\left(\frac{d}{2}\right)^2T.
\]

### Підсумковий об'єм

\[
V=V_{корп}+V_{ребер}-V_{отворів}.
\]

Усі розміри задаються в міліметрах, тому об'єм отримуємо в мм³. Для переведення в см³ ділимо на 1000.

In [ ]:
def calculate_geometry(W,H,D,T,n,d,E,G,Rt):
    if min(W,H,D,T,d,E,G,Rt) <= 0:
        raise ValueError("Усі параметри мають бути додатними.")
    if T >= min(H,D):
        raise ValueError("Товщина T має бути меншою за H і D.")
    if n < 1:
        raise ValueError("Кількість отворів n має бути не меншою за 1.")
    if E <= d/2 or (n > 1 and 2*E >= W):
        raise ValueError("Перевірте відступ E та діаметр отвору.")
    if G <= T or G >= min(H,D)-T:
        raise ValueError("Некоректний катет ребра G.")
    if Rt >= W:
        raise ValueError("Товщина ребра Rt має бути меншою за W.")

    r=d/2
    # Перевірка розташування отворів на полицях
    xhole=0.70*D
    yhole=0.68*H
    if xhole-r <= T or xhole+r >= D:
        raise ValueError("Отвір горизонтальної полиці наближається до межі/згину.")
    if yhole-r <= T or yhole+r >= H:
        raise ValueError("Отвір вертикальної полиці наближається до межі/згину.")

    zvals=np.array([W/2]) if n==1 else np.linspace(E,W-E,n)
    if n>1 and np.min(np.diff(zvals)) < d:
        raise ValueError("Отвори в ряду перетинаються: збільште W або зменште n чи d.")

    A_L=T*(D+H-T)
    V_body=A_L*W
    V_one_gusset=0.5*G*G*Rt
    V_gussets=2*V_one_gusset
    V_one_hole=np.pi*r*r*T
    V_holes=2*n*V_one_hole
    V_total=V_body+V_gussets-V_holes

    if V_total <= 0:
        raise ValueError("Розрахунковий об'єм має бути додатним.")

    return {
        "area_L_mm2":A_L,
        "body_mm3":V_body,
        "one_gusset_mm3":V_one_gusset,
        "gussets_mm3":V_gussets,
        "one_hole_mm3":V_one_hole,
        "holes_mm3":V_holes,
        "total_mm3":V_total,
        "total_cm3":V_total/1000,
        "hole_centers_z":zvals
    }

base = calculate_geometry(120,90,80,6,3,10,18,28,5)
base


## 2. Розрахунок маси для різних матеріалів

Використовуємо довідкові орієнтовні густини:
- конструкційна сталь — 7,85 г/см³;
- нержавіюча сталь — 7,90 г/см³;
- алюмінієвий сплав — 2,70 г/см³.

Маса:

\[
m=V_{см^3}\rho.
\]

Густина конкретної марки матеріалу може відрізнятися. Для навчального порівняння використовуємо наведені наближені значення.

In [ ]:
MATERIALS = {
    "Конструкційна сталь": 7.85,
    "Нержавіюча сталь": 7.90,
    "Алюмінієвий сплав": 2.70
}

def mass_table(geometry):
    rows=[]
    for name,rho in MATERIALS.items():
        rows.append({
            "Матеріал":name,
            "Густина, г/см³":rho,
            "Маса, г":geometry["total_cm3"]*rho,
            "Маса, кг":geometry["total_cm3"]*rho/1000
        })
    return pd.DataFrame(rows)

mass_table(base)


## 3. Структура об'єму

Порівняємо внесок корпусу, ребер жорсткості та віднімання матеріалу під отвори.

In [ ]:
def show_volume_breakdown(g):
    labels=["Основний корпус","Два ребра жорсткості","Матеріал отворів"]
    vals=[g["body_mm3"],g["gussets_mm3"],g["holes_mm3"]]
    fig,ax=plt.subplots(figsize=(9,5))
    bars=ax.bar(labels, np.array(vals)/1000)
    ax.set_ylabel("Об'єм, см³")
    ax.set_title("Складові розрахунку об'єму")
    ax.grid(axis="y",alpha=.3)
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,b.get_height(),
                f"{v/1000:.2f}",ha="center",va="bottom")
    plt.xticks(rotation=10)
    plt.tight_layout()
    plt.show()

show_volume_breakdown(base)


## 4. Інтерактивний розрахунок

Змінюйте параметри. Таблиця маси та діаграма оновлюватимуться автоматично.

In [ ]:
def interactive_calculation(W=120,H=90,D=80,T=6,n=3,d=10,E=18,G=28,Rt=5):
    try:
        g=calculate_geometry(W,H,D,T,n,d,E,G,Rt)
        print(f"Площа L-профілю: {g['area_L_mm2']:.1f} мм²")
        print(f"Об'єм корпусу: {g['body_mm3']/1000:.2f} см³")
        print(f"Об'єм двох ребер: {g['gussets_mm3']/1000:.2f} см³")
        print(f"Об'єм, вилучений отворами: {g['holes_mm3']/1000:.2f} см³")
        print(f"ПІДСУМКОВИЙ ОБ'ЄМ: {g['total_cm3']:.2f} см³")
        display(mass_table(g).round(4))
        show_volume_breakdown(g)
    except ValueError as e:
        print("Помилка параметрів:",e)

wW=widgets.IntSlider(value=120,min=80,max=180,step=5,description="W")
wH=widgets.IntSlider(value=90,min=60,max=150,step=5,description="H")
wD=widgets.IntSlider(value=80,min=60,max=140,step=5,description="D")
wT=widgets.IntSlider(value=6,min=3,max=12,step=1,description="T")
wn=widgets.IntSlider(value=3,min=1,max=6,step=1,description="n")
wd=widgets.IntSlider(value=10,min=6,max=16,step=1,description="d")
wE=widgets.IntSlider(value=18,min=12,max=30,step=1,description="E")
wG=widgets.IntSlider(value=28,min=15,max=45,step=1,description="G")
wRt=widgets.IntSlider(value=5,min=3,max=10,step=1,description="Rt")

controls=widgets.VBox([
    widgets.HBox([wW,wH,wD]),
    widgets.HBox([wT,wn,wd]),
    widgets.HBox([wE,wG,wRt])
])
out=widgets.interactive_output(interactive_calculation,{
    "W":wW,"H":wH,"D":wD,"T":wT,"n":wn,"d":wd,"E":wE,"G":wG,"Rt":wRt
})
display(controls,out)


## 5. Оцінка впливу конструктивних змін

Порівняємо три варіанти:
1. кронштейн без ребер жорсткості та без отворів;
2. кронштейн із ребрами, але без отворів;
3. кронштейн із ребрами та отворами.

Це дає змогу кількісно оцінити, скільки матеріалу додають ребра та скільки вилучається отворами.

In [ ]:
def compare_designs(W=120,H=90,D=80,T=6,n=3,d=10,E=18,G=28,Rt=5,
                    material="Конструкційна сталь"):
    g=calculate_geometry(W,H,D,T,n,d,E,G,Rt)
    rho=MATERIALS[material]
    V_plain=g["body_mm3"]
    V_with_gussets=V_plain+g["gussets_mm3"]
    V_final=g["total_mm3"]
    labels=["Корпус без ребер/отворів",
            "Корпус + ребра",
            "Корпус + ребра − отвори"]
    volumes=[V_plain,V_with_gussets,V_final]
    masses=[v/1000*rho for v in volumes]
    df=pd.DataFrame({
        "Варіант":labels,
        "Об'єм, см³":np.array(volumes)/1000,
        "Маса, г":masses
    })
    display(df.round(3))

    added_mass=g["gussets_mm3"]/1000*rho
    removed_mass=g["holes_mm3"]/1000*rho
    print(f"Маса, додана двома ребрами: {added_mass:.2f} г")
    print(f"Маса, вилучена отворами: {removed_mass:.2f} г")
    print(f"Чиста зміна маси відносно корпусу без ребер та отворів: "
          f"{(g['total_mm3']-g['body_mm3'])/1000*rho:+.2f} г")

compare_designs()


## 6. Самостійні завдання

1. Обчисліть масу початкового кронштейна зі сталі та з алюмінієвого сплаву. У скільки разів відрізняються результати?
2. Збільште `G` з 28 до 40 мм. На скільки змінилася маса?
3. Змініть кількість отворів `n` з 2 до 5. Визначте зміну об'єму та маси.
4. Порівняйте масу кронштейна з ребрами жорсткості та без них.
5. Підберіть параметри, за яких маса алюмінієвого кронштейна буде меншою за 0,20 кг, не порушуючи перевірок геометрії.

## 7. Контрольні запитання

1. Чому при обчисленні площі L-профілю потрібно відняти `T²`?
2. Чому об'єм трикутного ребра дорівнює `G²Rt/2`?
3. Чому для двох полиць кількість отворів дорівнює `2n`?
4. У яких одиницях потрібно подавати густину, щоб коректно отримати масу в грамах?
5. Які конструктивні елементи збільшують масу, а які зменшують?
6. Чому розрахункова маса не є точною масою готового виробу?

**Очікуваний результат:** студент може перейти від параметричної геометрії до кількісної оцінки матеріаломісткості та обґрунтувати вплив конструктивних рішень.